In [26]:
# ================================
# CELDA 1: Imports y Configuración
# ================================

import os
import cv2
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import shutil
from tqdm import tqdm

# Configuración de directorios
SOURCE_DIR = "datasets/plates_dirt"  # Cambia esta ruta
OUTPUT_DIR_POLY = "datasets/extracted_poly_coords_dirt/"   # Para bounding boxes grandes
OUTPUT_DIR_CHARS = "datasets/extracted_characters_dirt/"   # Para caracteres individuales

# Crear directorios de salida si no existen
Path(OUTPUT_DIR_POLY).mkdir(exist_ok=True)
Path(OUTPUT_DIR_CHARS).mkdir(exist_ok=True)

print("✅ Configuración completa")
print(f"📁 Directorio fuente: {SOURCE_DIR}")
print(f"📁 Directorio poly_coord: {OUTPUT_DIR_POLY}")
print(f"📁 Directorio characters: {OUTPUT_DIR_CHARS}")

✅ Configuración completa
📁 Directorio fuente: datasets/plates_dirt
📁 Directorio poly_coord: datasets/extracted_poly_coords_dirt/
📁 Directorio characters: datasets/extracted_characters_dirt/


In [27]:
# ================================
# CELDA 2: Funciones Auxiliares
# ================================

def load_image_and_json(base_name, source_dir):
    """Carga una imagen y su JSON correspondiente"""
    # Buscar la imagen (puede ser .jpg, .png, etc.)
    img_extensions = ['.jpg', '.jpeg', '.png', '.bmp']
    img_path = None
    
    for ext in img_extensions:
        potential_path = Path(source_dir) / f"{base_name}{ext}"
        if potential_path.exists():
            img_path = potential_path
            break
    
    if img_path is None:
        print(f"❌ No se encontró imagen para: {base_name}")
        return None, None
    
    # Cargar imagen
    image = cv2.imread(str(img_path))
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    # Cargar JSON
    json_path = Path(source_dir) / f"{base_name}.json"
    if not json_path.exists():
        print(f"❌ No se encontró JSON para: {base_name}")
        return image_rgb, None
    
    with open(json_path, 'r', encoding='utf-8') as f:
        json_data = json.load(f)
    
    return image_rgb, json_data

def draw_polygon(image, points, color, thickness=2, label=None):
    """Dibuja un polígono en la imagen"""
    image_copy = image.copy()
    
    # Convertir puntos a formato numpy
    points_np = np.array(points, dtype=np.int32)
    
    # Dibujar el polígono
    cv2.polylines(image_copy, [points_np], isClosed=True, color=color, thickness=thickness)
    
    # Añadir etiqueta si se proporciona
    if label:
        x, y = points_np[0]  # Posición de la etiqueta
        cv2.putText(image_copy, label, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    
    return image_copy

def extract_polygon_region(image, points):
    """Extrae la región definida por el polígono usando bounding box"""
    points_np = np.array(points, dtype=np.int32)
    
    # Obtener bounding box
    x, y, w, h = cv2.boundingRect(points_np)
    
    # Extraer región
    extracted_region = image[y:y+h, x:x+w]
    
    return extracted_region, (x, y, w, h)

def extract_bbox_region(image, bbox_coord):
    """Extrae la región definida por bbox_coord (dos puntos: esquina superior izq y inferior der)"""
    x1, y1 = bbox_coord[0]
    x2, y2 = bbox_coord[1]
    
    # Asegurar que x1,y1 sea la esquina superior izquierda
    x_min, x_max = min(x1, x2), max(x1, x2)
    y_min, y_max = min(y1, y2), max(y1, y2)
    
    # Extraer región
    extracted_region = image[y_min:y_max, x_min:x_max]
    
    return extracted_region, (x_min, y_min, x_max-x_min, y_max-y_min)

In [28]:
# ================================
# CELDA 3: Prueba de Visualización
# ================================

# Cambiar este nombre por uno de tus archivos (sin extensión)
TEST_FILE_NAME = "09 10007"  # ¡CAMBIA ESTE NOMBRE!

# Cargar imagen y JSON de prueba
test_image, test_json = load_image_and_json(TEST_FILE_NAME, SOURCE_DIR)

if test_image is not None and test_json is not None:
    print("✅ Carga exitosa!")
    print(f"📏 Tamaño de imagen: {test_image.shape}")
    print("📋 Estructura del JSON:")
    print(json.dumps(test_json, indent=2)[:500] + "..." if len(str(test_json)) > 500 else json.dumps(test_json, indent=2))
    
    # Mostrar imagen original
    plt.figure(figsize=(12, 8))
    plt.imshow(test_image)
    plt.title(f"Imagen Original: {TEST_FILE_NAME}")
    plt.axis('off')
    plt.show()
    
else:
    print("❌ Error al cargar archivos de prueba")
    print("🔧 Verifica:")
    print(f"   - Que existe {TEST_FILE_NAME}.json en {SOURCE_DIR}")
    print(f"   - Que existe {TEST_FILE_NAME}.jpg/png en {SOURCE_DIR}")

❌ No se encontró imagen para: 09 10007
❌ Error al cargar archivos de prueba
🔧 Verifica:
   - Que existe 09 10007.json en datasets/plates_dirt
   - Que existe 09 10007.jpg/png en datasets/plates_dirt


In [29]:
# ====================================================
# CELDA DE VISUALIZACIÓN CON OVERLAYS (COLOR Y TAMAÑO AJUSTADOS)
# ====================================================

# Asumimos que las siguientes variables ya existen en tu notebook:
# test_image: La imagen cargada con OpenCV y convertida a RGB
# test_json: El contenido del archivo JSON ya cargado
# TEST_FILE_NAME: El nombre del archivo de imagen para el título del plot

if test_image is not None and test_json is not None:
    # Crear copia de la imagen para dibujar sobre ella
    overlay_image = test_image.copy()
    
    # --- AJUSTES ---
    # 1. Definir un solo color: Azul Oscuro en formato (B, G, R) para OpenCV
    DARK_BLUE = (0, 0, 139)
    
    # 2. Ajustar el tamaño y grosor de la fuente
    FONT_SCALE = 0.65
    FONT_THICKNESS = 2
    # ----------------

    # Iterar directamente sobre la lista de objetos en el JSON
    for i, char_data in enumerate(test_json):
        if 'bbox_xywh' in char_data:
            # Obtener el caracter y la caja delimitadora
            char_id = char_data.get('char', '?')
            bbox_xywh = char_data['bbox_xywh']
            
            # El formato es [x, y, ancho, alto]
            x, y, w, h = bbox_xywh
            
            # Definir las esquinas del rectángulo
            top_left = (int(x), int(y))
            bottom_right = (int(x + w), int(y + h))
            
            # Dibujar el rectángulo en la imagen (siempre azul)
            cv2.rectangle(overlay_image, top_left, bottom_right, DARK_BLUE, thickness=2)
            
            # Escribir el caracter encima de la caja (siempre azul y más pequeño)
            label_pos = (top_left[0], top_left[1] - 10)
            cv2.putText(overlay_image, char_id, label_pos, cv2.FONT_HERSHEY_SIMPLEX, FONT_SCALE, DARK_BLUE, thickness=FONT_THICKNESS)
    
    print(f"✅ Se han dibujado {len(test_json)} caracteres.")
    
    # Mostrar la imagen resultante con los overlays
    plt.figure(figsize=(15, 10))
    plt.imshow(overlay_image)
    plt.title(f"Imagen con Overlays: {TEST_FILE_NAME}")
    plt.axis('off')
    plt.show()

else:
    print("❌ Asegúrate de que las variables 'test_image' y 'test_json' estén cargadas.")

❌ Asegúrate de que las variables 'test_image' y 'test_json' estén cargadas.


In [30]:
# ==========================================================
# CELDA DE PROCESAMIENTO Y EXTRACCIÓN (UNIFICADA Y CORREGIDA)
# ==========================================================

from pathlib import Path
from tqdm import tqdm
import cv2
import json
import matplotlib.pyplot as plt

# --- 1. FUNCIÓN DE EXTRACCIÓN DE CARACTERES (CORREGIDA) ---

def extract_characters(image, json_data, output_dir, file_name):
    """
    Extrae y guarda cada carácter basándose en la nueva estructura de JSON.
    La nueva estructura es una lista directa de caracteres.
    """
    extracted_count = 0
    
    # Iteramos directamente sobre la lista del JSON, ya no buscamos 'lps'
    for i, char_info in enumerate(json_data):
        if 'bbox_xywh' not in char_info or 'char' not in char_info:
            continue
        
        # Leemos los nuevos nombres de clave: 'char' y 'bbox_xywh'
        bbox_xywh = char_info['bbox_xywh']
        char_id = char_info.get('char')
        
        # Si el char_id está vacío o es inválido, lo omitimos
        if not char_id:
            continue
            
        # Crear directorio para este caracter si no existe
        char_dir = Path(output_dir) / str(char_id)
        char_dir.mkdir(exist_ok=True)
        
        try:
            # Extraer la región usando el formato [x, y, w, h]
            x, y, w, h = map(int, bbox_xywh)
            extracted_char = image[y : y+h, x : x+w]
            
            # Verificar que la región extraída no esté vacía
            if extracted_char.size == 0:
                continue
            
            # Guardar el caracter recortado
            output_path = char_dir / f"{file_name}_char_{char_id}_{i}.png"
            cv2.imwrite(str(output_path), extracted_char)
            
            extracted_count += 1
            
        except Exception as e:
            continue
    
    return extracted_count


def order_points(pts):
    """
    Ordena 4 puntos de manera robusta basándose en el centroide.
    Retorna: [TL, TR, BR, BL]
    """
    # Calcular el centroide
    centroid = pts.mean(axis=0)
    
    # Separar puntos en izquierda/derecha y arriba/abajo respecto al centroide
    left_points = []
    right_points = []
    
    for pt in pts:
        if pt[0] < centroid[0]:
            left_points.append(pt)
        else:
            right_points.append(pt)
    
    # Ordenar izquierda por Y (arriba primero)
    left_points = sorted(left_points, key=lambda p: p[1])
    # Ordenar derecha por Y (arriba primero)
    right_points = sorted(right_points, key=lambda p: p[1])
    
    # Asignar: TL, BL de izquierda; TR, BR de derecha
    tl = left_points[0] if len(left_points) > 0 else pts[0]
    bl = left_points[1] if len(left_points) > 1 else pts[3]
    tr = right_points[0] if len(right_points) > 0 else pts[1]
    br = right_points[1] if len(right_points) > 1 else pts[2]
    
    return np.array([tl, tr, br, bl], dtype=np.float32)


def extract_plate_strip(image, json_data):
    """
    Versión mejorada que endereza la placa correctamente sin importar su orientación.
    """
    try:
        # 1. Recolectar todos los puntos de todas las cajas de caracteres
        all_points = []
        valid_annotations = [item for item in json_data if item.get('char', ' ').strip()]
        if len(valid_annotations) < 2:
            return None
            
        for item in valid_annotations:
            x, y, w, h = item['bbox_xywh']
            all_points.extend([[x, y], [x + w, y], [x, y + h], [x + w, y + h]])

        # 2. Encontrar el rectángulo de área mínima
        contour = np.array(all_points, dtype=np.int32)
        rect = cv2.minAreaRect(contour)
        box_points = cv2.boxPoints(rect)
        
        # 3. Obtener el ángulo del rectángulo
        angle = rect[2]
        width, height = rect[1]
        
        # 4. Ajustar el ángulo y dimensiones para que la placa quede horizontal
        # Si el ancho es menor que el alto, la placa está rotada 90 grados
        if width < height:
            angle = angle + 90
            width, height = height, width
        
        # 5. Normalizar el ángulo
        if angle < -45:
            angle = angle + 90
        elif angle > 45:
            angle = angle - 90
            
        # 6. Ordenar los puntos correctamente
        pts_source_sorted = order_points(box_points)
        
        # 7. Calcular dimensiones de salida más precisas
        output_width = int(width)
        output_height = int(height)
        
        # Asegurar dimensiones mínimas
        if output_width < 10 or output_height < 10:
            return None
        
        # 8. Definir el rectángulo de destino (horizontal)
        pts_dest = np.float32([
            [0, 0],
            [output_width - 1, 0],
            [output_width - 1, output_height - 1],
            [0, output_height - 1]
        ])
        
        # 9. Aplicar transformación de perspectiva
        matrix = cv2.getPerspectiveTransform(pts_source_sorted, pts_dest)
        plate_strip = cv2.warpPerspective(image, matrix, (output_width, output_height))
        
        # 10. Verificar si la placa quedó rotada (más alta que ancha)
        # Si es así, rotarla 90 grados
        if plate_strip.shape[0] > plate_strip.shape[1]:
            plate_strip = cv2.rotate(plate_strip, cv2.ROTATE_90_CLOCKWISE)
        
        return plate_strip

    except Exception as e:
        print(f"Error extrayendo la tira de la placa: {e}")
        return None


# --- 2. FUNCIÓN DE PROCESAMIENTO COMPLETO (SIMPLIFICADA) ---

def process_complete_dataset(source_dir, chars_output_dir, strips_output_dir):
    """
    Procesa todo el dataset, extrayendo tanto los caracteres individuales
    como la tira de placa completa.
    """
    
    # --- Preparación ---
    json_files = list(Path(source_dir).glob('*.json'))
    
    if not json_files:
        print(f"❌ No se encontraron archivos JSON en {source_dir}")
        return
    
    # Crear directorios de salida
    Path(chars_output_dir).mkdir(exist_ok=True)
    Path(strips_output_dir).mkdir(exist_ok=True)
    
    print(f"🔍 Encontrados {len(json_files)} archivos JSON para procesar.")
    
    char_total = 0
    strip_total = 0
    errors = []
    
    # --- Bucle de Procesamiento ---
    for json_file in tqdm(json_files, desc="Procesando archivos"):
        base_name = json_file.stem
        
        # Cargar imagen y JSON
        try:
            with open(json_file, 'r') as f:
                json_data = json.load(f)

            # Encontrar la imagen correspondiente
            img_path = None
            for ext in ['.jpg', '.jpeg', '.png', '.bmp']:
                potential_path = json_file.with_suffix(ext)
                if potential_path.exists():
                    img_path = potential_path
                    break
            
            if not img_path:
                errors.append(f"No se encontró imagen para {base_name}")
                continue

            image = cv2.imread(str(img_path))
            if image is None:
                errors.append(f"Error cargando imagen {base_name}")
                continue
                
        except Exception as e:
            errors.append(f"Error cargando datos para {base_name}: {str(e)}")
            continue

        # --- Tarea 1: Extraer caracteres individuales (usando tu función 'extract_characters') ---
        # (Asumimos que la función 'extract_characters' ya está definida en una celda anterior)
        char_count = extract_characters(image, json_data, chars_output_dir, base_name)
        char_total += char_count
        
        # --- Tarea 2: Extraer la tira de placa completa (usando 'extract_plate_strip') ---
        # (Asumimos que la función 'extract_plate_strip' ya está definida en una celda anterior)
        extracted_strip = extract_plate_strip(image, json_data)
        
        if extracted_strip is not None:
            # Obtener el texto de la placa para el nombre del archivo
            plate_text = "".join([item.get('char', '') for item in json_data]).strip()
            
            # Guardar la tira de placa
            output_path = Path(strips_output_dir) / f"{plate_text}.jpg"
            cv2.imwrite(str(output_path), extracted_strip)
            strip_total += 1
    
    # --- Resumen Final ---
    print("\n" + "="*50)
    print("📊 RESUMEN DE PROCESAMIENTO")
    print("="*50)
    print(f"✅ Tiras de placa completas extraídas: {strip_total}")
    print(f"✅ Caracteres individuales extraídos: {char_total}")
    print(f"❌ Archivos con errores: {len(errors)}")
    
    if errors:
        print("\n🔍 Primeros 10 errores:")
        for error in errors[:10]:
            print(f"  - {error}")
            
# --- 3. EJECUCIÓN ---
# (Asumimos que OUTPUT_DIR_CHARS y SOURCE_DATA_DIR están definidos)
# process_complete_dataset(SOURCE_DATA_DIR, OUTPUT_DIR_CHARS)

# --- 4. VISUALIZACIÓN DE PRUEBA ---
# (Usando las variables 'test_image', 'test_json' que ya tenías cargadas)
if 'test_image' in locals() and 'test_json' in locals() and test_image is not None and test_json is not None:
    print("\nEjecutando visualización de prueba...")
    # Creamos un directorio temporal para la visualización
    temp_char_dir = Path("./temp_extracted_chars/")
    temp_char_dir.mkdir(exist_ok=True)

    extract_characters(cv2.cvtColor(test_image, cv2.COLOR_RGB2BGR), test_json, temp_char_dir, "test_file")

    char_dirs = [d for d in temp_char_dir.iterdir() if d.is_dir()]
    if char_dirs:
        plt.figure(figsize=(15, 5))
        num_to_show = min(8, len(char_dirs))
        for i, char_dir in enumerate(sorted(char_dirs)[:num_to_show]):
            char_files = list(char_dir.glob('*.png'))
            if char_files:
                char_image = cv2.imread(str(char_files[0]))
                char_image_rgb = cv2.cvtColor(char_image, cv2.COLOR_BGR2RGB)
                plt.subplot(1, num_to_show, i + 1)
                plt.imshow(char_image_rgb)
                plt.title(f"Clase: '{char_dir.name}'")
                plt.axis('off')
        plt.tight_layout()
        plt.show()

In [31]:
# ================================
# CELDA 8: Ejecutar Procesamiento Completo
# ================================

# ¡DESCOMENTA ESTA LÍNEA CUANDO ESTÉS LISTO PARA PROCESAR TODO!
process_complete_dataset(SOURCE_DIR, OUTPUT_DIR_CHARS, OUTPUT_DIR_POLY)

print("🚀 Para procesar todo el dataset, descomenta la línea anterior y ejecuta esta celda")
print("⚠️  Asegúrate primero de que la celda de prueba funciona correctamente")

🔍 Encontrados 5050 archivos JSON para procesar.


Procesando archivos:   0%|          | 0/5050 [00:00<?, ?it/s]

Procesando archivos: 100%|██████████| 5050/5050 [02:08<00:00, 39.44it/s]


📊 RESUMEN DE PROCESAMIENTO
✅ Tiras de placa completas extraídas: 5050
✅ Caracteres individuales extraídos: 38891
❌ Archivos con errores: 0
🚀 Para procesar todo el dataset, descomenta la línea anterior y ejecuta esta celda
⚠️  Asegúrate primero de que la celda de prueba funciona correctamente
